# This is a sample Jupyter Notebook

Below is an example of a code cell. 
Put your cursor into the cell and press Shift+Enter to execute it and select the next one, or click 'Run Cell' button.

Press Double Shift to search everywhere for classes, files, tool windows, actions, and settings.

To learn more about Jupyter Notebooks in PyCharm, see [help](https://www.jetbrains.com/help/pycharm/ipython-notebook-support.html).
For an overview of PyCharm, go to Help -> Learn IDE features or refer to [our documentation](https://www.jetbrains.com/help/pycharm/getting-started.html).

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets, transforms

import numpy as np
import cv2

import tkinter as tk

# --- Incarc MNIST si normalizez ---
transform = transforms.Compose([
    transforms.ToTensor(),  # transforma la [0,1], shape (1,28,28)
])

train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform, split='byclass')
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform, split='byclass')

x_train = train_dataset.data.numpy()
y_train = train_dataset.targets.numpy()
x_test = test_dataset.data.numpy()
y_test = test_dataset.targets.numpy()

# Normalizez la [0,1]
x_train = x_train / 255.0
x_test = x_test / 255.0

In [2]:
from digitRecogn2.geometric_props import calculate_geometric_features

# Calcul geometric features pentru seturi
geo_train = np.array([calculate_geometric_features(img) for img in x_train])
geo_test = np.array([calculate_geometric_features(img) for img in x_test])

# Convertesc totul in torch tensors
x_train_tensor = torch.tensor(x_train).unsqueeze(1).float()  # (N, 1, 28, 28)
geo_train_tensor = torch.tensor(geo_train).float()
y_train_tensor = torch.tensor(y_train).long()

x_test_tensor = torch.tensor(x_test).unsqueeze(1).float()
geo_test_tensor = torch.tensor(geo_test).float()
y_test_tensor = torch.tensor(y_test).long()

# Creez DataLoader (opțional)
train_loader = DataLoader(torch.utils.data.TensorDataset(x_train_tensor, geo_train_tensor, y_train_tensor),
                          batch_size=64, shuffle=True)
test_loader = DataLoader(torch.utils.data.TensorDataset(x_test_tensor, geo_test_tensor, y_test_tensor),
                         batch_size=64, shuffle=False)

In [ ]:
from digitRecogn2.model_arch import CNNGeoModel

def train_model(model, train_loader, epochs=30):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters())

    model.train()
    for epoch in range(epochs):
        running_loss = 0.0
        correct = 0
        total = 0
        for imgs, geos, labels in train_loader:
            imgs, geos, labels = imgs.to(device), geos.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(imgs, geos)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        print(f"Epoch {epoch+1}/{epochs} - Loss: {running_loss/len(train_loader):.4f} - Accuracy: {100*correct/total:.2f}%")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CNNGeoModel(num_classes=10).to(device)

# Pentru antrenare, rulează:
train_model(model, train_loader, epochs=10)

# Salvare model după antrenare
torch.save(model.state_dict(), './models/m1.pth')


Epoch 1/10 - Loss: 0.6962 - Accuracy: 78.13%
Epoch 2/10 - Loss: 0.4433 - Accuracy: 84.56%
Epoch 3/10 - Loss: 0.4076 - Accuracy: 85.59%
Epoch 4/10 - Loss: 0.3855 - Accuracy: 86.18%
Epoch 5/10 - Loss: 0.3706 - Accuracy: 86.61%


In [16]:
def test_model(model, test_loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for imgs, geos, labels in test_loader:
            imgs, geos, labels = imgs.to(device), geos.to(device), labels.to(device)
            outputs = model(imgs, geos)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    print(f"Test Accuracy: {100 * correct / total:.2f}%")

# Pentru testare
model.load_state_dict(torch.load('./models/m1.pth'))
test_model(model, test_loader)


Test Accuracy: 85.89%


In [17]:
label_map = datasets.EMNIST(root='./data', split='byclass', download=True).classes
class DrawingApp:
    def __init__(self, root, model):
        self.root = root
        self.model = model
        self.model.eval()

        self.root.title("Desenează o cifră")

        self.canvas = tk.Canvas(root, width=280, height=280, bg='white')
        self.canvas.pack()
        self.canvas.bind("<B1-Motion>", self.paint)

        self.image = np.zeros((280, 280), dtype=np.uint8)
        self.drawing = False

        self.predict_button = tk.Button(root, text="Recunoaște", command=self.recognize)
        self.predict_button.pack()

        self.clear_button = tk.Button(root, text="Șterge", command=self.clear)
        self.clear_button.pack()

        self.prediction_label = tk.Label(root, text="Cifra prezisă: -", font=("Helvetica", 16))
        self.prediction_label.pack()

        self.accuracy_label = tk.Label(root, text="Probabilitate: -", font=("Helvetica", 14))
        self.accuracy_label.pack()

    def paint(self, event):
        x, y = event.x, event.y
        r = 8
        self.canvas.create_oval(x - r, y - r, x + r, y + r, fill='black')
        if 0 <= x < 280 and 0 <= y < 280:
            self.image[y-r:y+r, x-r:x+r] = 255
            self.drawing = True

    def clear(self):
        self.canvas.delete("all")
        self.image.fill(0)
        self.prediction_label.config(text="Cifra prezisă: -")
        self.accuracy_label.config(text="Probabilitate: -")
        self.drawing = False

    def recognize(self):
        if not self.drawing:
            return

        # Redimensionez la 28x28 si normalizez
        img28 = cv2.resize(self.image, (28, 28))
        img28 = img28.astype(np.float32) / 255.0

        # Calculez trăsături geometrice si scalez
        geo = np.array(calculate_geometric_features(img28)).reshape(1, -1)
        geo_scaled = self.scaler.transform(geo)

        # Convert la tensor torch
        img_tensor = torch.tensor(img28).unsqueeze(0).unsqueeze(0).float().to(device)  # (1,1,28,28)
        geo_tensor = torch.tensor(geo_scaled).float().to(device)  # (1,7)

        with torch.no_grad():
            outputs = self.model(img_tensor, geo_tensor)
            probs = torch.softmax(outputs, dim=1)
            prob_val, pred = torch.max(probs, 1)
            pred = label_map[pred.item()]

        self.prediction_label.config(text=f"Cifra prezisă: {pred}")
        self.accuracy_label.config(text=f"Probabilitate: {prob_val.item()*100:.2f}%")
        self.drawing = False

# Pentru a porni aplicația:

model = CNNGeoModel(num_geo_features=7, num_classes=10)
model.load_state_dict(torch.load('./models/m1.pth', map_location=device))
model.to(device)

root = tk.Tk()
app = DrawingApp(root, model)
